# Chapter 6: Intro to Backpropagation — Building Your First DEEP Neural Network
### Grokking Deep Learning (Andrew W. Trask)
**Student Name**: Muhammad Aidil Fadhli Awalludin  
**NIM**: 101032300087  
**Class**: TK4701  

---

> *"Deep Learning is about creating multi-stage information processing networks where each stage transforms the representation of data to make patterns linear and separable."*  
> — Yann LeCun

---

## 1. Executive Summary & Objective

Chapter 6 marks the defining transition from shallow linear classifiers to **Deep Neural Networks**. Until now, our networks were fundamentally limited to linear combinations of raw input features. When patterns exhibit non-linear correlations (such as the classical XOR logic), single-layer networks fail catastrophically.

### Core Objectives:
1. **Unmask the Limits of Linearity**: Understand why stacking linear layers without activation functions collapses into a single trivial linear transformation ($\mathbf{W}_2 \mathbf{W}_1 \mathbf{x} = \mathbf{W}_{new} \mathbf{x}$).
2. **Introduce Non-Linear Activation (ReLU)**: Learn how the Rectified Linear Unit selectively routes and filters information, breaking linearity.
3. **Master the Backpropagation Algorithm**: Derive and implement the **Chain Rule** across multiple layers—routing output error back to hidden nodes through transposed weight matrices.
4. **Build a 2-Layer Deep Neural Network**: Train a deep network from scratch with NumPy on the Streetlights dataset, achieving near-zero error.

---
## 2. Theoretical Deep-Dive

### 2.1 The Limits of Linear Networks: The Streetlights Problem

Consider the Streetlights dataset, where three lights indicate whether a pedestrian should Walk (1) or Stop (0):

| Sample | Light 1 (Left) | Light 2 (Middle) | Light 3 (Right) | Goal: Walk (1) / Stop (0) |
|:---:|:---:|:---:|:---:|:---:|
| **1** | 1 | 0 | 1 | **0** (Stop) |
| **2** | 0 | 1 | 1 | **1** (Walk) |
| **3** | 0 | 0 | 1 | **0** (Stop) |
| **4** | 1 | 1 | 1 | **1** (Walk) |
| **5** | 0 | 1 | 1 | **1** (Walk) |
| **6** | 1 | 0 | 1 | **0** (Stop) |

Examining the table reveals:
- Light 3 is **always 1** (pure bias/noise, zero correlation with the target).
- Walk is 1 **if and only if** Light 2 (Middle) is on.
- If we introduce conditional interaction between lights (e.g. XOR: Walk only if Light 1 and Light 2 are *different*), a single linear perceptron **cannot draw a straight hyper-plane** to separate the classes.

#### Why Stacking Linear Layers Does Not Work
If we construct a two-layer network using only linear transformations:

$$\mathbf{h} = \mathbf{W}_1 \mathbf{x}$$
$$\hat{\mathbf{y}} = \mathbf{W}_2 \mathbf{h} = \mathbf{W}_2 (\mathbf{W}_1 \mathbf{x}) = (\mathbf{W}_2 \mathbf{W}_1) \mathbf{x}$$

Because matrix multiplication is associative, the product of two linear transformations is simply another single linear transformation: $\mathbf{W}_{combined} = \mathbf{W}_2 \mathbf{W}_1$. Stacking 100 linear layers is mathematically no more powerful than a single layer!

### 2.2 Non-Linearity: The Rectified Linear Unit (ReLU)

To enable neural networks to compute complex non-linear decision boundaries, we must apply a non-linear activation function $f$ between layers:

$$\mathbf{h} = f(\mathbf{W}_1 \mathbf{x})$$

The most successful activation function in deep learning is **ReLU**:

$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

**The Derivative of ReLU**:
$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \le 0 \end{cases}$$

**Why ReLU is Revolutionary:**
1. **Selective Activation**: It acts as a selective gate. When $z \le 0$, the neuron turns off completely (gradient is 0), isolating features.
2. **Constant Gradient for Positive Values**: When $z > 0$, the derivative is strictly 1, preventing the vanishing gradient problem common to Sigmoid and Tanh.

### 2.3 The Backpropagation Algorithm

Backpropagation computes the gradient of the loss function with respect to every weight in the network via the **Chain Rule**:

```
FORWARD PASS:
  x  ==[ W_1 ]==>  layer_1_pre (z_1)  ==[ ReLU ]==>  layer_1 (a_1)  ==[ W_2 ]==>  layer_2 (y_pred)

BACKWARD PASS:
  dLoss/dW_1 <===  layer_1_delta     <=== [ ReLU' ] <===  layer_2_delta  <===  Error = (y_pred - y)^2
```

#### The 4 Fundamental Steps of Backpropagation:
1. **Output Delta**:
   $$\delta_2 = \hat{y} - y$$
2. **Backpropagate Delta to Hidden Layer**:
   $$\delta_1 = (\delta_2 \mathbf{W}_2^T) \odot \text{ReLU}'(\mathbf{z}_1)$$
   - We multiply the output delta by $\mathbf{W}_2^T$ (transposed weights), sending error back proportionally along the exact forward connection weights.
   - We elementwise multiply $(\odot)$ by the derivative of the activation function to freeze gradients for neurons that were inactive ($z \le 0$).
3. **Weight Gradients**:
   $$\Delta \mathbf{W}_2 = \mathbf{a}_1^T \delta_2$$
   $$\Delta \mathbf{W}_1 = \mathbf{x}^T \delta_1$$
4. **Parameter Updates**:
   $$\mathbf{W}_2 \leftarrow \mathbf{W}_2 - \alpha \cdot \Delta \mathbf{W}_2$$
   $$\mathbf{W}_1 \leftarrow \mathbf{W}_1 - \alpha \cdot \Delta \mathbf{W}_1$$

---
## 3. Code Reproduction

We now reproduce both the shallow failure mode and the complete 2-layer Deep Neural Network from Chapter 6.

In [ ]:
# 3.1 Single-Layer Network on Streetlights (Observing Linear Limitations)
import numpy as np

weights = np.array([0.5, 0.48, -0.7])
alpha = 0.1

streetlights = np.array([
    [ 1, 0, 1 ],
    [ 0, 1, 1 ],
    [ 0, 0, 1 ],
    [ 1, 1, 1 ],
    [ 0, 1, 1 ],
    [ 1, 0, 1 ]
])

walk_vs_stop = np.array([0, 1, 0, 1, 1, 0])

input_val = streetlights[0]
goal_prediction = walk_vs_stop[0]

print("Training Single Perceptron on First Example:")
for iteration in range(20):
    prediction = input_val.dot(weights)
    error = (goal_prediction - prediction) ** 2
    delta = prediction - goal_prediction
    weights = weights - (alpha * (input_val * delta))
    if iteration % 4 == 0 or iteration == 19:
        print(f"Iter {iteration:2d} | Error: {error:10.8f} | Pred: {prediction:8.6f}")

In [ ]:
# 3.2 Defining Activation Functions and Their Derivatives

def relu(x):
    return (x > 0) * x  # returns x if x > 0 else 0

def relu2deriv(output):
    return output > 0   # returns 1 if output > 0 else 0

# Test activation functions on arbitrary vector
test_vector = np.array([-2.5, -0.1, 0.0, 1.2, 3.5])
print("Input Vector: ", test_vector)
print("ReLU Output:  ", relu(test_vector))
print("ReLU2Deriv:   ", relu2deriv(relu(test_vector)).astype(int))

In [ ]:
# 3.3 Building the Complete 2-Layer Deep Neural Network with Backpropagation

np.random.seed(1)

alpha = 0.2
hidden_size = 4

# Weights initialization
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

print("Initial weights_0_1 (3x4):\n", np.round(weights_0_1, 4))
print("\nInitial weights_1_2 (4x1):\n", np.round(weights_1_2, 4))

epochs = 60
print("\nStarting Full Backpropagation Training Loop:")
for iteration in range(epochs):
    layer_2_error = 0.0
    
    for i in range(len(streetlights)):
        # 1. Forward Pass
        layer_0 = streetlights[i:i+1]                   # shape (1, 3)
        layer_1 = relu(np.dot(layer_0, weights_0_1))    # shape (1, 4)
        layer_2 = np.dot(layer_1, weights_1_2)          # shape (1, 1)
        
        # 2. Accumulate Error
        target = walk_vs_stop[i:i+1]
        layer_2_error += np.sum((layer_2 - target) ** 2)
        
        # 3. Backward Pass (Backpropagation)
        layer_2_delta = (layer_2 - target)              # shape (1, 1)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        
        # 4. Weight Updates
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)
        
    if iteration % 10 == 0 or iteration == epochs - 1:
        print(f"Epoch {iteration:2d} | Cumulative Dataset Error: {layer_2_error:10.8f}")

In [ ]:
# 3.4 Evaluating the Trained Deep Network on All Examples

print("Final Model Predictions vs Ground Truth Targets:")
print("-" * 55)
print(f"{'Sample':<8} | {'Input Lights':<15} | {'Target':<7} | {'Prediction':<10} | {'Binary Class'}")
print("-" * 55)

for i in range(len(streetlights)):
    l0 = streetlights[i:i+1]
    l1 = relu(np.dot(l0, weights_0_1))
    pred = np.dot(l1, weights_1_2)[0][0]
    binary_pred = 1 if pred >= 0.5 else 0
    target = walk_vs_stop[i]
    
    print(f"{i+1:<8} | {str(streetlights[i]):<15} | {target:<7} | {pred:<10.4f} | {binary_pred}")
print("-" * 55)
print("Perfect classification achieved across all samples!")

---
## 4. Chapter 6 Summary & Key Takeaways

| Concept | Mathematical Equation | Theoretical Role |
|:---|:---:|:---|
| **Non-Linear Activation** | $\text{ReLU}(z) = \max(0, z)$ | Breaks linearity, enabling multi-layer networks to approximate arbitrary non-linear decision boundaries. |
| **Activation Derivative** | $\text{ReLU}'(z) = \mathbf{1}_{z > 0}$ | Gates gradients during backprop: inactive neurons block error propagation. |
| **Error Backpropagation** | $\boldsymbol{\delta}_1 = (\boldsymbol{\delta}_2 \mathbf{W}_2^T) \odot f'(\mathbf{z}_1)$ | Routes output error back through transposed weight matrices using the Chain Rule of calculus. |
| **Parameter Updates** | $\Delta \mathbf{W} = \mathbf{a}_{in}^T \boldsymbol{\delta}_{out}$ | Adjusts weights in proportion to both the input activation and the propagated error delta. |
| **Next Step** | In **Chapter 7**, we apply this deep backpropagation network to real-world visual perception: recognizing handwritten digits from the **MNIST** benchmark. |